### This recipe was developed in the context of a set of supervised fine-tuning experiments. The use case involves a parquet file with a particular column that identifies different sub-categories of data and all the rows containing one of these sub-categories need to be removed. DPK filter transform is used for achieving this. The resulting parquet file needs to be converted to a jsonl format. 


##### **** These pip installs need to be adapted to use the appropriate release level. Alternatively, The venv running the jupyter lab could be pre-configured with a requirement file that includes the right release. If running from git clone:
```
python -m venv venv
source venv/bin/activate 
pip install jupyterlab
venv/bin/jupyter lab


In [ ]:
%%capture
# Users and application developers must use the right tag for the latest from pypi
%pip install "data-prep-toolkit-transforms[filter]==1.1.5"
%pip install python-dotenv

##### **** Import required classes and modules

In [ ]:
from dpk_filter.runtime import Filter

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
input_folder= os.getenv("INPUT_FOLDER", "input")
output_folder=os.getenv("OUTPUT_FOLDER", "output")


##### **** Setup runtime parameters and invoke the transform. For a full list of parameters for the filter transform, please see [here](./README.md).

In [ ]:

Filter(input_folder=input_folder, 
       output_folder=output_folder,
        filter_columns_to_drop= [],
        filter_criteria_list= ['_meta_json != \'{"dataset": "numinamath", "source": "olympiads", "split": "train"}\''],
        filter_doc_id_column_name = "uuid",
      ).transform()

##### **** The specified folder will include the transformed parquet files.

In [ ]:
import glob
output_files=glob.glob(f"{output_folder}/*.parquet")

Convert all parquet files in the output folder to jsonl 

In [ ]:
%%time
import pyarrow.parquet as pq
import json

for parquet_file_path in output_files:
    table = pq.read_table(parquet_file_path)
    pylist=table.to_pylist()
    json_file=f"{os.path.splitext(parquet_file_path)[0]}.jsonl"
#    ## write records one line at a time
    with open(json_file, 'w') as f:
        for d in pylist:
            json.dump(d, f)
            f.write('\n')

